In [43]:
import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm

def pair_data(y, x, start_date="2018-01-01"):
    print(f"Fetching data for {y} and {x}")
    data = yf.download([y, x], start=start_date)['Close']
    return data.ffill().dropna()

In [45]:
def rolling_hedge(Y, X, window=60):

    beta = []

    for i in range(len(Y)):
        if i < window:
            beta.append(np.nan)
        else:
            y = Y.iloc[i-window:i]
            x = X.iloc[i-window:i]
            model = sm.OLS(y, sm.add_constant(x)).fit()
            beta.append(model.params.iloc[1])

    return pd.Series(beta, index=Y.index).ffill()

In [47]:
def spread(y, x, beta_t):
    return y - beta_t * x

In [49]:
def ou(spread):

    lag = spread.shift(1).dropna()
    diff = spread.diff().dropna()

    lag = lag.loc[diff.index]

    X = sm.add_constant(lag)
    model = sm.OLS(diff, X).fit()

    lam = model.params.iloc[1]
    mu = -model.params.iloc[0] / lam
    sigma = np.std(model.resid)

    if lam < 0:
        half_life = -np.log(2) / lam
    else:
        half_life = np.inf

    print(f"OU λ: {lam:.5f}")
    print(f"OU μ: {mu:.4f}")
    print(f"Half-Life: {half_life:.1f} days")

    return lam, mu, sigma, half_life

In [51]:
def ou_zscore(spread, mu, sigma, lam):

    sigma_eq = sigma / np.sqrt(-2 * lam)
    z = (spread - mu) / sigma_eq

    return z.dropna()

In [53]:
def score_signals(z):

    signals_df = pd.DataFrame(index=z.index)
    signals_df['Z_Score'] = z

    signals_df['Long_Spread'] = z < -2.0
    signals_df['Short_Spread'] = z > 2.0

    signals_df['Exit'] = z.abs() < 0.5
    signals_df['Stop_Loss'] = z.abs() > 5.0

    print(f"Signals Generated: {signals_df['Long_Spread'].sum()} Long, {signals_df['Short_Spread'].sum()} Short")

    return signals_df

In [55]:
def turnover(signals_df):

    target = pd.Series(np.nan, index=signals_df.index)

    target.loc[signals_df['Long_Spread']] = 1.0
    target.loc[signals_df['Short_Spread']] = -1.0

    target.loc[signals_df['Exit']] = 0.0
    target.loc[signals_df['Stop_Loss']] = 0.0

    target = target.ffill().fillna(0.0)

    actual = target.shift(1).fillna(0.0)

    trades = actual.diff().abs().sum()

    print(f"Trades Executed: {int(trades)}")

    return actual

In [57]:
def cost_backtest(prices_df, target, beta_t, y, x, cost_bps=10):

    print(f"Friction Model: {cost_bps} bps")

    ret_y = prices_df[y].pct_change().fillna(0)
    ret_x = prices_df[x].pct_change().fillna(0)

    notional = 1 + abs(beta_t)

    spread_return = (ret_y - beta_t * ret_x) / notional

    gross_returns = target * spread_return

    turnover = target.diff().abs().fillna(0)
    cost = turnover * (cost_bps / 10000)

    net_returns = gross_returns - cost
    equity_curve = (1 + net_returns).cumprod()

    total_return = equity_curve.iloc[-1] - 1

    print(f"Net Return: {total_return:.2%}")

    return equity_curve, net_returns

In [59]:
def performance(returns):

    sharpe = np.mean(returns) / np.std(returns) * np.sqrt(252)

    equity = (1 + returns).cumprod()
    drawdown = (equity / equity.cummax() - 1).min()

    print(f"Sharpe: {sharpe:.2f}")
    print(f"Max Drawdown: {drawdown:.2%}")

    return sharpe, drawdown

In [61]:
if __name__ == "__main__":

    Y_TICKER = "SHEL"
    X_TICKER = "BP"

    prices = pair_data(Y_TICKER, X_TICKER)

    Y = np.log(prices[Y_TICKER])
    X = np.log(prices[X_TICKER])

    beta_t = rolling_hedge(Y, X, window=60)

    spread_val = spread(Y, X, beta_t)

    lam, mu, sigma, half_life = ou(spread_val)

    z = ou_zscore(spread_val, mu, sigma, lam)

    signals = score_signals(z)

    target = turnover(signals)

    prices = prices.loc[target.index]
    beta_t = beta_t.loc[target.index]

    equity, returns = cost_backtest(
        prices, target, beta_t, Y_TICKER, X_TICKER, cost_bps=10
    )

    performance(returns)

Fetching data for SHEL and BP


[*********************100%***********************]  2 of 2 completed


OU λ: -0.00346
OU μ: 1.6149
Half-Life: 200.4 days
Signals Generated: 67 Long, 33 Short
Trades Executed: 12
Friction Model: 10 bps
Net Return: 33.67%
Sharpe: 0.77
Max Drawdown: -6.26%
